In [ ]:
import html

import ipywidgets as widgets
from sage.all import *
from IPython.display import display, clear_output, HTML, Math

# Load the public Sage solver and warm the generator cache.
display(HTML("<div style='color:#666; font-weight:bold;'>Loading core library...</div>"))
load("sagecode/solver_core.sage")
preload_generator_chunks_async()
clear_output(wait=True)

# ==============================================================================
# Notebook Rendering Helpers
# ==============================================================================

MAX_NOTEBOOK_MATH_RENDER_CHARS = 8000
HEAVY_RENDER_LEVELS = {131}


def should_render_math(level, latex_text):
    """Avoid full MathJax rendering for extremely large formulas."""
    return level not in HEAVY_RENDER_LEVELS and len(latex_text) <= MAX_NOTEBOOK_MATH_RENDER_CHARS


def render_math_with_codebox(lhs, rhs_latex, title=None, render_math=True):
    """Render one formula and keep the raw LaTeX available in a copyable text box."""
    if lhs:
        full_latex = f"{lhs} = {rhs_latex}"
    else:
        full_latex = rhs_latex

    box_layout = widgets.Layout(
        border='1px solid #d0d0d0',
        padding='15px',
        margin='0 0 20px 0',
        border_radius='8px',
        width='98%'
    )

    container = widgets.Output(layout=box_layout)
    with container:
        if title:
            display(HTML(
                f"<div style='font-weight:600; margin-bottom:8px; font-family:sans-serif;'>"
                f"{html.escape(title)}</div>"
            ))
        if render_math:
            display(Math(full_latex))
        else:
            if lhs:
                display(Math(lhs))
            display(HTML(
                "<div style='color:#7a5c00; background:#fff8e1; border:1px solid #ecd58b; "
                "border-radius:6px; padding:10px 12px; margin-bottom:10px; font-family:sans-serif;'>"
                "Math rendering was skipped for this formula to keep the notebook responsive."
                "</div>"
            ))
        html_code = f"""
        <div style="margin-top: 15px; border-top: 1px dashed #eee; padding-top: 10px;">
            <div style="font-size: 0.85em; color: #666; margin-bottom: 5px; font-weight: bold; font-family: sans-serif;">
                Raw LaTeX code (click to select):
            </div>
            <textarea readonly
                onclick="this.select();"
                style="
                    width: 100%;
                    height: 50px;
                    padding: 8px;
                    font-family: 'Courier New', monospace;
                    font-size: 0.9em;
                    background-color: #f8f9fa;
                    border: 1px solid #ccc;
                    border-radius: 4px;
                    resize: vertical;
                    box-sizing: border-box;
                ">{html.escape(full_latex)}</textarea>
        </div>
        """
        display(HTML(html_code))
    display(container)


def render_formula(level, lhs, rhs_latex, title=None):
    """Render one report formula with the notebook-specific large-formula safeguard."""
    full_latex = f"{lhs} = {rhs_latex}" if lhs else rhs_latex
    render_math_with_codebox(lhs, rhs_latex, title=title, render_math=should_render_math(level, full_latex))


def render_family_report(report):
    """Render the structured report payload returned by solve_isogeny_report."""
    N = report['N']
    display(HTML(f"<h2 style='border-bottom: 2px solid #333; padding-bottom: 10px;'>Parametric Family of Cyclic {N}-Isogenies</h2>"))
    display(Math(report['base_curve_note']))
    render_formula(N, '', report['base_eq'])

    if report['family_type'] == 'hyperelliptic':
        display(HTML("<div style='margin: 10px 0 20px 0; color: #444;'>Fixed hyperelliptic case from the quadratic-point classification.</div>"))
        display(HTML(r"<h3 style='color: #0055aa; border-left: 5px solid #0055aa; padding-left: 10px;'>Hyperelliptic Discriminant</h3>"))
        render_formula(N, report['hyperelliptic']['discriminant_label'], report['hyperelliptic']['discriminant'])
        display(Math(report['hyperelliptic']['note']))

    display(HTML(r"<h3 style='color: #0055aa; border-left: 5px solid #0055aa; padding-left: 10px;'>1. Domain Curve</h3>"))
    for lhs, rhs in report['domain_coeffs']:
        render_formula(N, lhs, rhs)

    display(HTML(r"<h3 style='color: #0055aa; border-left: 5px solid #0055aa; padding-left: 10px; margin-top: 30px;'>2. Codomain Curve</h3>"))
    for lhs, rhs in report['codomain_coeffs']:
        render_formula(N, lhs, rhs)

    if report['family_type'] == 'bielliptic':
        info = report['bielliptic']
        display(HTML(r"<h3 style='color: #0055aa; border-left: 5px solid #0055aa; padding-left: 10px; margin-top: 30px;'>3. Bielliptic Quotient</h3>"))
        display(Math(r"\textbf{Elliptic Quotient: } " + info['label']))
        render_formula(N, '', info['equation'])
        render_formula(N, r'x_E(t,\alpha_t)', info['x_map'])
        display(Math(r"\mathrm{rank}\,E(\mathbf{Q}) = " + str(info['rank'])))
        if info['generators']:
            gen_text = ', '.join([latex(P) for P in info['generators']])
            display(Math(r"\textbf{Generators of } E(\mathbf{Q}):\ " + gen_text))
        if info['distinguished_generator'] is not None:
            display(Math(r"P_1 = " + latex(info['distinguished_generator'])))
            display(Math(r"P_n = nP_1 = (x_n,y_n)"))
        display(HTML(r"<h3 style='color: #0055aa; border-left: 5px solid #0055aa; padding-left: 10px; margin-top: 30px;'>4. Quadratic Family Over the Quotient</h3>"))
        render_formula(N, '', info['sequence_quadratic_relation'])
        render_formula(N, '', info['sequence_linear_relation'])
        display(Math(
            rf"\text{{For each }} n \in \mathbf{{Z}} \text{{ with }} P_n=(x_n,y_n)\in {info['label']}(\mathbf{{Q}}),"
            rf"\text{{ the coordinate }} {info['primitive_var']} \text{{ is quadratic over }} \mathbf{{Q}}."
        ))
        if info['sample_points']:
            display(HTML(r"<h3 style='color: #0055aa; border-left: 5px solid #0055aa; padding-left: 10px; margin-top: 30px;'>5. Sample Quadratic Points</h3>"))
            for sample in info['sample_points']:
                P = sample['quotient_point']
                display(Math(rf"P_{{{sample['multiple']}}} = {latex(P)}"))
                for idx, pt in enumerate(sample['fiber_points'], start=1):
                    field_poly = pt['field'].defining_polynomial() if hasattr(pt['field'], 'defining_polynomial') else None
                    if field_poly is not None:
                        field_text = latex(field_poly)
                        display(Math(rf"K_{{{sample['multiple']},{idx}}} = \mathbf{{Q}}[z]/({field_text})"))
                    display(Math(rf"(x,y)_{{{sample['multiple']},{idx}}} = \left({latex(pt['x'])}, {latex(pt['y'])}\right)"))


def run_isogeny_construction(N, prec=500, strict_degree_chain=None, factor_output=True):
    """Run one level through solve_isogeny_report and render the resulting report."""
    init_message = f"Initializing computation for N={N}..."
    if N == 131:
        init_message = "Initializing specialized computation for N=131..."

    status_label = widgets.HTML(
        value=f"<div style='color:blue; font-weight:bold;'>{init_message}</div>"
    )
    display(status_label)

    def ui_progress(label, deg, max_deg, y_label, work_prec):
        status_label.value = (
            f"<div style='color:blue; font-weight:bold;'>"
            f"Computing rational map for <b>{label}</b>... <br>"
            f"Current Degree Search: <b>{deg}</b> (Max: {max_deg}), "
            f"max_deg_y = <b>{y_label}</b>, prec = <b>{work_prec}</b>"
            f"</div>"
        )

    try:
        report = solve_isogeny_report(
            N,
            prec=prec,
            progress_cb=ui_progress,
            strict_degree_chain=strict_degree_chain,
            factor_output=factor_output,
        )
    except NotImplementedError as e:
        clear_output()
        display(HTML(f"<div style='color:red'><b>Error:</b> {html.escape(str(e))}</div>"))
        return

    if report is None:
        clear_output()
        display(HTML("<div style='color:red'><b>Failed:</b> Could not find all rational maps.</div>"))
        return

    clear_output()
    render_family_report(report)


n_list = [11, 14, 15, 17]
n_list += list(range(19, 25))
n_list += list(range(26, 34))
n_list += [35, 36, 37, 39, 40, 41, 43, 46, 47, 48, 49, 50, 53, 59, 61, 65, 71, 79, 83, 89, 101, 131]
n_list.sort()

header = widgets.HTML("<h2>Explicit Construction of Cyclic Isogenies</h2>")
label_n = widgets.HTML(value="<b>Select Level N:</b>", layout=widgets.Layout(margin='5px 10px 0 0'))
n_selector = widgets.Dropdown(options=n_list, value=11, layout=widgets.Layout(width='100px'))
run_button = widgets.Button(description='Compute', button_style='primary', icon='play', layout=widgets.Layout(width='120px', margin='0 0 0 10px'))
controls = widgets.HBox([label_n, n_selector, run_button], layout=widgets.Layout(align_items='center', margin='0 0 10px 0'))
out = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='15px'))


def on_button_clicked(b):
    """Clear the output area and run the selected level."""
    out.clear_output()
    with out:
        try:
            run_isogeny_construction(n_selector.value)
        except Exception as e:
            print(f"An error occurred: {e}")
            import traceback
            traceback.print_exc()


run_button.on_click(on_button_clicked)
display(widgets.VBox([header, controls, out]))



In [ ]:
# Batch helpers are provided by sagecode/solver_core.sage.
# Example manual calls:
# report = solve_single_case_report(43)
# run_batch_and_save([43, 61, 83])
